In [ ]:
pip install xgboost

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold

# ==============================================================================
# 0. CONFIGURACIÓN DEL EXPERIMENTO Y VALIDACIÓN CRUZADA
# ==============================================================================
experiment_name = "/modelos-hipervalorizacion-bogota-v2"
mlflow.set_experiment(experiment_name)

N_SPLITS = 5
RANDOM_STATE = 42
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
matriz = "/Volumes/workspace/default/datos_bogota/matriz_barrios_unificada.csv"
df = pd.read_csv(matriz)

# Mismo subconjunto supervisado e índice reseteado para garantizar pliegues idénticos
df_supervised = df[df["v_ref_2026"].notnull()].copy().reset_index(drop=True)

y_sup = df_supervised["v_ref_2026"].copy()
cols_a_excluir = [
    "barrio", "geometry", "v_ref_2015", "v_ref_2026",
    "num_listings_airbnb", "area_km2"
]
X_sup = df_supervised.drop(
    columns=[col for col in cols_a_excluir if col in df_supervised.columns]
).copy()

# XGBoost requiere variables numéricas
X_sup = pd.get_dummies(X_sup, columns=["localidad"], drop_first=True, dtype=int)

# Mejores hiperparámetros según MLflow
params = {
    "colsample_bytree": 0.8,
    "learning_rate": 0.05,
    "max_depth": 3,
    "n_estimators": 300,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "reg_lambda": 5,
    "subsample": 0.8,
}


def log_cv_metrics(metricas_dict: dict) -> None:
    """Calcula media, desviación estándar y brecha de sobreajuste, y las registra en MLflow."""
    resumen = {
        "cv_r2_train_mean": float(np.mean(metricas_dict["r2_train"])),
        "cv_r2_train_std": float(np.std(metricas_dict["r2_train"])),
        "cv_r2_val_mean": float(np.mean(metricas_dict["r2_val"])),
        "cv_r2_val_std": float(np.std(metricas_dict["r2_val"])),
        "cv_r2_gap_mean": float(
            np.mean(np.array(metricas_dict["r2_train"]) - np.array(metricas_dict["r2_val"]))
        ),
        "cv_mae_train_mean": float(np.mean(metricas_dict["mae_train"])),
        "cv_mae_val_mean": float(np.mean(metricas_dict["mae_val"])),
        "cv_mae_val_std": float(np.std(metricas_dict["mae_val"])),
        "cv_rmse_val_mean": float(np.mean(metricas_dict["rmse_val"])),
        "cv_rmse_val_std": float(np.std(metricas_dict["rmse_val"])),
    }
    mlflow.log_metrics(resumen)
    for k, v in resumen.items():
        print(f"  {k}: {v:,.4f}")


def ejecutar_experimento_xgb(
    run_name: str, artifact_path: str, usar_log: bool = False
) -> None:
    """Ejecuta validación cruzada K-Fold, registra métricas, gráfico y modelo final."""
    print(f"\n--- EVALUANDO {run_name} ---")
    metricas = {
        "r2_train": [], "r2_val": [],
        "mae_train": [], "mae_val": [], "rmse_val": []
    }

    for train_idx, val_idx in kf.split(X_sup):
        X_tr, X_val = X_sup.iloc[train_idx], X_sup.iloc[val_idx]
        y_tr, y_val = y_sup.iloc[train_idx], y_sup.iloc[val_idx]

        y_tr_fit = np.log(y_tr) if usar_log else y_tr

        modelo_fold = xgb.XGBRegressor(**params)
        modelo_fold.fit(X_tr, y_tr_fit)

        y_pred_tr = modelo_fold.predict(X_tr)
        y_pred_val = modelo_fold.predict(X_val)

        if usar_log:
            y_pred_tr = np.exp(y_pred_tr)
            y_pred_val = np.exp(y_pred_val)

        metricas["r2_train"].append(r2_score(y_tr, y_pred_tr))
        metricas["r2_val"].append(r2_score(y_val, y_pred_val))
        metricas["mae_train"].append(mean_absolute_error(y_tr, y_pred_tr))
        metricas["mae_val"].append(mean_absolute_error(y_val, y_pred_val))
        metricas["rmse_val"].append(np.sqrt(mean_squared_error(y_val, y_pred_val)))

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            **params,
            "modelo": "XGBRegressor",
            "features": "todas_con_v_ref_2025",
            "target_transform": "Logaritmo Natural" if usar_log else "Ninguna",
            "cv_splits": N_SPLITS,
        })

        log_cv_metrics(metricas)

        # Entrenamiento final sobre todo el conjunto supervisado para artefacto e importancia
        y_final_fit = np.log(y_sup) if usar_log else y_sup
        modelo_final = xgb.XGBRegressor(**params)
        modelo_final.fit(X_sup, y_final_fit)

        # Gráfico de importancia de variables (Top 15)
        importancias = modelo_final.feature_importances_
        top_15_idx = np.argsort(importancias)[-15:]
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh(range(len(top_15_idx)), importancias[top_15_idx], color="teal")
        ax.set_yticks(range(len(top_15_idx)))
        ax.set_yticklabels(X_sup.columns[top_15_idx])
        sufijo = "Log" if usar_log else "Bruto"
        ax.set_title(f"Top 15 Variables Más Importantes - XGBoost ({sufijo})")
        plt.tight_layout()
        mlflow.log_figure(fig, f"importancia_variables_{sufijo.lower()}.png")
        plt.close(fig)

        mlflow.xgboost.log_model(
            xgb_model=modelo_final,
            artifact_path=artifact_path,
            input_example=X_sup.head(5),
        )


# ==============================================================================
# RUN 1: XGBOOST SIN LOGARITMO (Precios Brutos)
# ==============================================================================
ejecutar_experimento_xgb(
    run_name="XGB_Sin_Log",
    artifact_path="xgboost_model_bruto",
    usar_log=False,
)

# ==============================================================================
# RUN 2: XGBOOST CON LOGARITMO
# ==============================================================================
ejecutar_experimento_xgb(
    run_name="XGB_Con_Log",
    artifact_path="xgboost_model_log",
    usar_log=True,
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import shap
import xgboost as xgb
from sklearn.inspection import PartialDependenceDisplay

# 1. Entrenar el modelo ganador en memoria (sin abrir ningún run en MLflow)
modelo_final = xgb.XGBRegressor(**params)
modelo_final.fit(X_sup, y_sup)

VARIABLE_INTERES = "densidad_airbnb_km2"

# 2. Calcular valores SHAP
explainer = shap.TreeExplainer(modelo_final)
shap_values = explainer.shap_values(X_sup)

idx_airbnb = X_sup.columns.get_loc(VARIABLE_INTERES)
shap_airbnb = shap_values[:, idx_airbnb]
valores_airbnb = X_sup[VARIABLE_INTERES].to_numpy()

# 3. Imprimir métricas cuantitativas de peso, dirección y percentiles
peso_medio_abs_cop = float(np.mean(np.abs(shap_airbnb)))
efecto_min_cop = float(np.min(shap_airbnb))
efecto_max_cop = float(np.max(shap_airbnb))
correlacion_direccion = float(np.corrcoef(valores_airbnb, shap_airbnb)[0, 1])
p90, p95, p99, max_obs = np.percentile(valores_airbnb, [90, 95, 99, 100])

print(f"--- EXPLICABILIDAD PARA {VARIABLE_INTERES} ---")
print(f"Peso medio absoluto (|SHAP|): ${peso_medio_abs_cop:,.2f} COP/m2")
print(f"Rango de impacto SHAP:        ${efecto_min_cop:,.2f} a ${efecto_max_cop:,.2f} COP/m2")
print(f"Correlación (dirección):      {correlacion_direccion:.4f}")
print(f"Percentiles observados -> P90: {p90:.2f} | P95: {p95:.2f} | P99: {p99:.2f} | Max: {max_obs:.2f}")

# 4. Gráfico 1: SHAP Summary Plot (Top 15)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sup, max_display=15, show=False)
plt.title("Impacto Global de Variables sobre el Valor del m² (SHAP - XGBoost)")
plt.tight_layout()
plt.show()

# 5. Gráfico 2: SHAP Dependence Plot para densidad_airbnb_km2
fig_dep, ax_dep = plt.subplots(figsize=(9, 6))
shap.dependence_plot(
    VARIABLE_INTERES,
    shap_values,
    X_sup,
    interaction_index="auto",
    ax=ax_dep,
    show=False,
)
ax_dep.set_title(f"Efecto Marginal SHAP de {VARIABLE_INTERES} sobre el Valor del m² (COP)")
ax_dep.set_ylabel("Contribución SHAP (COP / m²)")
plt.tight_layout()
plt.show()

# 6. Gráfico 3: Partial Dependence Plot (PDP) acotado al percentil 99
fig_pdp, ax_pdp = plt.subplots(figsize=(9, 6))
PartialDependenceDisplay.from_estimator(
    estimator=modelo_final,
    X=X_sup,
    features=[VARIABLE_INTERES],
    percentiles=(0.0, 0.99),
    grid_resolution=50,
    ax=ax_pdp,
)
ax_pdp.set_title(f"Gráfico de Dependencia Parcial (PDP) - {VARIABLE_INTERES}")
ax_pdp.set_xlabel("Densidad de alojamientos Airbnb (alojamientos / km²)")
ax_pdp.set_ylabel("Valor de referencia predicho 2026 (COP / m²)")
ax_pdp.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()